# Selective next-range-bar direction research

This notebook consumes the reusable `range_bars_ml` library; feature construction and training logic do not live here.


In [13]:
# Reload local modelling modules so re-running this cell picks up source edits.
import importlib
import range_bars_ml as _range_bars_package
import range_bars_ml.model as _range_bars_model
import range_bars_ml.pipeline as _range_bars_pipeline
importlib.reload(_range_bars_model)
importlib.reload(_range_bars_pipeline)
importlib.reload(_range_bars_package)

from range_bars_ml import (
    ExperimentConfig,
    FeatureConfig,
    ModelConfig,
    load_range_bars,
    prepare_dataset,
    run_experiment,
)
from range_bars_ml.model import signal_metrics
import numpy as np
import polars as pl


In [23]:
# Data selection
DATA_PATH = r"C:\Code\Trading\2026\ml\data\renko_bars_10.parquet"

# Causal feature definition
FEATURE_CONFIG = FeatureConfig(
    windows=range(20),
    time_column="close_time",
)

# LightGBM and probability-calibration definition
MODEL_CONFIG = ModelConfig(
    calibration_fraction=0.10,
    random_state=256,
    n_estimators=200,
    learning_rate=0.03,
    num_leaves=100,
)

# Require a probability edge above the development-period up rate.
# Set to 0.0 to disable this additional anti-trend gate.
EDGE_MARGIN = 0.02

# Chronological evaluation and selective-signal definition
EXPERIMENT_CONFIG = ExperimentConfig(
    feature=FEATURE_CONFIG,
    model=MODEL_CONFIG,
    test_fraction=0.20,
    n_folds=3,
    # Zero permits an all-abstain strategy if no calls have positive net score.
    minimum_coverage=0.0,
    abstention_penalty=-0.001,
    edge_margin=EDGE_MARGIN,
)

# Inspection and artifact choices
FEATURE_PREVIEW_ROWS = 10
FEATURE_EXPORT_PATH = None  # e.g. r"C:\Code\Trading\2026\ml\data\range_bar_features.parquet"
FEATURE_EXPORT_COMPRESSION = "zstd"
MODEL_ARTIFACT_PATH = "range_bar_model.joblib"


In [42]:
bars = load_range_bars(DATA_PATH)
bars = bars.filter(pl.col("close_time") < pl.datetime(2025, 1, 1, time_zone='UTC'))
#bars = bars.slice(offset=DATA_SLICE_OFFSET, length=DATA_SLICE_LENGTH)


In [43]:
bars.shape

(5330154, 12)

In [45]:
# Inspect the fully causal, next-bar-labelled training frame.
feature_df, feature_names = prepare_dataset(bars, config=FEATURE_CONFIG)
feature_df.head(FEATURE_PREVIEW_ROWS)

# Set FEATURE_EXPORT_PATH above to persist the complete feature frame.
if FEATURE_EXPORT_PATH:
    feature_df.write_parquet(FEATURE_EXPORT_PATH, compression=FEATURE_EXPORT_COMPRESSION)


In [46]:
# Metrics include score (+1 correct, -1 wrong, -0.001 abstain).
result = run_experiment(bars, config=EXPERIMENT_CONFIG)

# The edge gate is applied inside run_experiment before its single final-test score.
dataset, _ = prepare_dataset(bars, config=FEATURE_CONFIG)
test = dataset[result.split_plan.test.tolist()]
development_up_rate = result.model.metadata["development_up_rate"]


In [47]:
# Compare against simple regime/trend baselines on the same final test.
test_target = test["target"].to_numpy()
baseline_metrics = {
    "always_up": signal_metrics(np.ones(len(test_target)), test_target, 0.5, -0.01, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
    "always_down": signal_metrics(np.zeros(len(test_target)), test_target, 1.01, 0.5, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
    "always_abstain": signal_metrics(np.full(len(test_target), 0.5), test_target, 1.01, -0.01, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
}
score_comparison = pl.DataFrame([
    {"strategy": "model_edge_gated", **result.test_metrics},
    *({"strategy": name, **metrics} for name, metrics in baseline_metrics.items()),
]).select(["strategy", "score", "mean_score", "signals", "correct", "wrong", "abstentions", "precision"])

print(f"Development up rate: {development_up_rate:.3%}; edge gates: long >= {result.model.long_threshold:.3f}, short <= {result.model.short_threshold:.3f}")
score_comparison

Development up rate: 49.716%; edge gates: long >= 0.517, short <= 0.477


strategy,score,mean_score,signals,correct,wrong,abstentions,precision
str,f64,f64,f64,f64,f64,f64,f64
"""model_edge_gated""",59934.015,0.056222,906044.0,483069.0,422975.0,159985.0,0.533163
"""always_up""",9075.0,0.008513,1.066029e6,537552.0,528477.0,0.0,0.504256
"""always_down""",-9075.0,-0.008513,1.066029e6,528477.0,537552.0,0.0,0.495744
"""always_abstain""",-1066.029,-0.001,0.0,0.0,0.0,1.066029e6,NaN


In [33]:
result.model.save(MODEL_ARTIFACT_PATH)


In [49]:
bars['close_time']

close_time
"datetime[μs, UTC]"
2017-08-17 04:02:48.879 UTC
2017-08-17 04:03:48.038 UTC
2017-08-17 04:21:13.349 UTC
2017-08-17 04:28:39.149 UTC
2017-08-17 04:29:53.911 UTC
…
2024-12-31 23:33:09.928 UTC
2024-12-31 23:36:33.115 UTC
2024-12-31 23:50:47.615 UTC
